# Replication of Andrews et al. (2022): The Returns to College Major Choice

Project by Qilin Li, Spring 2026

---

This notebook contains my replication of the results from the following paper:

> Andrews, R. J., Imberman, S. A., Lovenheim, M. F., & Stange, K. M. (2022).  
> The Returns to College Major Choice: Average and Distributional Effects, Career Trajectories, and Earnings Variability.  
> NBER Working Paper No. 30331.

### Table of Contents

1. Introduction  
2. Data  
3. Empirical Framework  
4. Replication of Andrews et al. (2022)  
   4.1 Descriptive Statistics  
   4.2 Mean Earnings Differences Across Majors  
   4.3 Earnings by Major Category  
   4.4 Baseline Regression Results  
5. Extension and Robustness  
   5.1 Alternative Specifications  
   5.2 Log Earnings Specification  
6. Discussion  
7. Conclusion  
8. References

In [2]:
import pandas as pd

## 1. Introduction

Differences in earnings across college majors are large and persistent in the U.S. labor market. Students’ choice of major is therefore one of the most consequential educational decisions they make. Understanding whether these differences reflect skill acquisition, labor market demand, or selection into fields of study is central to education and labor economics.

Andrews et al. (2022) examine the returns to college major choice using detailed administrative data from Texas. The authors document substantial variation in average earnings across majors and further analyze how these differences vary across the earnings distribution, over career trajectories, and in terms of earnings variability. Their findings suggest that major choice plays an important role in shaping both short-run and long-run labor market outcomes.

In this notebook, I replicate the core results of Andrews et al. (2022), focusing on differences in mean earnings across college majors. Using publicly available data from the American Community Survey (ACS), I estimate the association between major choice and median earnings and compare the patterns observed in this dataset to the findings reported in the original paper. While this replication does not reproduce the distributional or lifecycle analyses conducted by the authors, it aims to assess whether the central result — that earnings differ substantially across majors — is robust in alternative data.

### Main Variables

The main variables used in this replication are summarized below.

In [2]:
main_vars = pd.DataFrame(
    {
        "Category": [
            "Outcome Variable",
            "Main Explanatory Variable",
            "Control Variable",
        ],
        "Variable": [
            "Median Earnings",
            "College Major / Major Category",
            "Unemployment Rate",
        ],
        "Description": [
            "Median annual earnings by college major",
            "Field of study (individual major or grouped category)",
            "Unemployment rate by major",
        ],
    }
)

main_vars

,Category,Variable,Description
0,Outcome Variable,Median Earnings,Median annual earnings by college major
1,Main Explanatory Variable,College Major / Major Category,Field of study (individual major or grouped ca...
2,Control Variable,Unemployment Rate,Unemployment rate by major


## 2. Data

This replication uses the “College Majors” dataset derived from the American Community Survey (ACS) 2010–2012. The dataset reports labor market outcomes by college major, including median earnings, unemployment rates, and employment composition.

The unit of observation is a college major. Each observation represents aggregated labor market outcomes for individuals with that field of study.

The primary outcome variable is **Median earnings**, which measures annual median earnings by major. The key explanatory variable is **college major** (or major category).

In [10]:
df = pd.read_csv("data/all-ages.csv")
df.head()

,Major_code,Major,Major_category,Total,Employed,Employed_full_time_year_round,Unemployed,Unemployment_rate,Median,P25th,P75th
0,1100,GENERAL AGRICULTURE,Agriculture & Natural Resources,128148,90245,74078,2423,0.026147,50000,34000,80000.0
1,1101,AGRICULTURE PRODUCTION AND MANAGEMENT,Agriculture & Natural Resources,95326,76865,64240,2266,0.028636,54000,36000,80000.0
2,1102,AGRICULTURAL ECONOMICS,Agriculture & Natural Resources,33955,26321,22810,821,0.030248,63000,40000,98000.0
3,1103,ANIMAL SCIENCES,Agriculture & Natural Resources,103549,81177,64937,3619,0.042679,46000,30000,72000.0
4,1104,FOOD SCIENCE,Agriculture & Natural Resources,24280,17281,12722,894,0.049188,62000,38500,90000.0


In [11]:
df.shape
df.describe()

,Major_code,Total,Employed,Employed_full_time_year_round,Unemployed,Unemployment_rate,Median,P25th,P75th
count,173.000000,1.730000e+02,1.730000e+02,1.730000e+02,173.000000,173.000000,173.000000,173.000000,173.000000
mean,3879.815029,2.302566e+05,1.661620e+05,1.263078e+05,9725.034682,0.057355,56816.184971,38697.109827,82506.358382
std,1687.753140,4.220685e+05,3.073244e+05,2.424254e+05,18022.040192,0.019177,14706.226865,9414.524761,20805.330126
min,1100.000000,2.396000e+03,1.492000e+03,1.093000e+03,0.000000,0.000000,35000.000000,24900.000000,45800.000000
25%,2403.000000,2.428000e+04,1.728100e+04,1.272200e+04,1101.000000,0.046261,46000.000000,32000.000000,70000.000000
50%,3608.000000,7.579100e+04,5.656400e+04,3.961300e+04,3619.000000,0.054719,53000.000000,36000.000000,80000.000000
75%,5503.000000,2.057630e+05,1.428790e+05,1.110250e+05,8862.000000,0.069043,65000.000000,42000.000000,95000.000000
max,6403.000000,3.123510e+06,2.354398e+06,1.939384e+06,147261.000000,0.156147,125000.000000,78000.000000,210000.000000


The dataset contains 173 college majors. Median earnings vary substantially across majors. 
The average median earnings are approximately $56,800, with values ranging from about $35,000 to $125,000. 
This large dispersion suggests that college major choice is strongly associated with labor market outcomes.

In [12]:
df.sort_values("Median", ascending=False)[["Major", "Median"]].head(10)

,Major,Median
59,PETROLEUM ENGINEERING,125000
154,PHARMACY PHARMACEUTICAL SCIENCES AND ADMINISTR...,106000
57,NAVAL ARCHITECTURE AND MARINE ENGINEERING,97000
55,METALLURGICAL ENGINEERING,96000
58,NUCLEAR ENGINEERING,95000
56,MINING AND MINERAL ENGINEERING,92000
97,MATHEMATICS AND COMPUTER SCIENCE,92000
48,ELECTRICAL ENGINEERING,88000
45,CHEMICAL ENGINEERING,86000
51,GEOLOGICAL AND GEOPHYSICAL ENGINEERING,85000


The highest-earning majors are concentrated in engineering and quantitative fields. 
Petroleum Engineering has the highest median earnings ($125,000), followed by Pharmacy and several engineering disciplines. 
These results suggest that STEM-related majors are associated with substantially higher labor market returns.

In [13]:
df.sort_values("Median")[["Major", "Median"]].head(10)

,Major,Median
88,NEUROSCIENCE,35000
31,EARLY CHILDHOOD EDUCATION,35300
145,STUDIO ARTS,37600
124,HUMAN SERVICES AND COMMUNITY ORGANIZATION,38000
117,COUNSELING PSYCHOLOGY,39000
77,LIBRARY SCIENCE,40000
28,ELEMENTARY EDUCATION,40000
24,COSMETOLOGY SERVICES AND CULINARY ARTS,40000
115,EDUCATIONAL PSYCHOLOGY,40000
74,COMPOSITION AND RHETORIC,40000


In contrast, the lowest-earning majors are concentrated in education, arts, and social service fields. 
Median earnings in these majors range between approximately $35,000 and $40,000. 
The gap between the highest- and lowest-paying majors exceeds $90,000, indicating substantial heterogeneity in labor market returns across fields of study.

In [14]:
df.groupby("Major_category")["Median"].mean().sort_values(ascending=False)

Major_category
Engineering                            77758.620690
Computers & Mathematics                66272.727273
Physical Sciences                      62400.000000
Business                               60615.384615
Health                                 56458.333333
Agriculture & Natural Resources        55000.000000
Social Science                         53222.222222
Law & Public Policy                    52800.000000
Industrial Arts & Consumer Services    52642.857143
Biology & Life Science                 50821.428571
Communications & Journalism            49500.000000
Humanities & Liberal Arts              46080.000000
Psychology & Social Work               44555.555556
Education                              43831.250000
Arts                                   43525.000000
Interdisciplinary                      43000.000000
Name: Median, dtype: float64

At the major category level, Engineering has the highest average median earnings (approximately $77,800), followed by Computers & Mathematics and Physical Sciences. 
In contrast, Education and Arts-related fields have substantially lower average earnings, around $43,000–$44,000. 

These descriptive patterns confirm that earnings differ widely across fields of study, consistent with the central findings of Andrews et al. (2022).

## 3. Empirical Framework

Differences in earnings across college majors can be understood through the lens of human capital theory. 
According to this framework, individuals invest in education to acquire skills that increase productivity and future earnings. 
Different majors provide different types of skills, which may be valued differently in the labor market.

Earnings differences may arise from several mechanisms. First, some fields of study may generate higher productivity due to technical or quantitative skill acquisition. 
Second, labor market demand may differ across sectors, leading to wage premiums in high-demand industries such as engineering or computing. 
Third, selection may play a role, as students with higher ability or stronger quantitative backgrounds may disproportionately choose certain majors.

Andrews et al. (2022) examine whether these earnings differences persist across the earnings distribution and over the lifecycle. 
In this replication, I focus on the central empirical finding that median earnings differ substantially across majors.

To examine the relationship between college major and earnings, I estimate a simple cross-sectional regression model:

$$
\log(Median_i) = \alpha + \beta MajorCategory_i + \epsilon_i
$$

where:

- $Median_i$ denotes median earnings for major $i$,
- $MajorCategory_i$ represents major category indicators,
- $\beta$ captures average differences in earnings across major categories.

This specification allows us to quantify how earnings differ systematically across fields of study.